# PiShield — correcting predictions at inference time

This notebook runs **end-to-end with no external downloads**. It shows how to use a
PiShield *Shield Layer* to correct the outputs of a neural network so that they are
*guaranteed* to satisfy a set of requirements.

We will:
1. write a small set of linear requirements,
2. build a Shield Layer from them,
3. apply it to some deliberately-invalid predictions and verify the result.

In [ ]:
import torch
from pishield.shield_layer import build_shield_layer

## 1. Define the requirements

A requirements file starts with an `ordering` line listing the variables, followed by one
linear inequality per line. Here we use 3 variables and require the output to be
**non-increasing**: `y_0 >= y_1 >= y_2`.

In [ ]:
requirements_path = "monotonic_requirements.txt"
with open(requirements_path, "w") as f:
    f.write("ordering y_0 y_1 y_2\n")  # order in which variables are corrected
    f.write("y_0 - y_1 >= 0\n")        # y_0 >= y_1
    f.write("y_1 - y_2 >= 0\n")        # y_1 >= y_2

print(open(requirements_path).read())

## 2. Build the Shield Layer

`build_shield_layer(num_variables, requirements_path)` parses the file and returns a
differentiable `torch.nn.Module`. The requirement *type* (linear here) is auto-detected.

In [ ]:
num_variables = 3
shield_layer = build_shield_layer(num_variables, requirements_path)

## 3. Correct some predictions

Imagine these came from a neural network. Both rows violate `y_0 >= y_1 >= y_2`.

In [ ]:
predictions = torch.tensor([
    [0.2, 0.9, 0.5],   # not monotone: 0.2 < 0.9
    [1.0, 0.0, 0.3],   # not monotone: 0.0 < 0.3
])

corrected = shield_layer(predictions.clone())

print("original predictions:\n", predictions)
print("\ncorrected predictions:\n", corrected)

## 4. Verify the corrected predictions satisfy the requirements

In [ ]:
def is_monotone_non_increasing(t, tol=1e-6):
    return bool((t[:, 0] >= t[:, 1] - tol).all() and (t[:, 1] >= t[:, 2] - tol).all())

print("original  satisfy requirements?", is_monotone_non_increasing(predictions))
print("corrected satisfy requirements?", is_monotone_non_increasing(corrected))

### Notes
- The Shield Layer is **differentiable**, so it can be embedded in a network and used at
  training time too — see `training.ipynb`.
- PiShield also supports **QFLRA** and **propositional** requirements. Either rely on
  auto-detection or pass `requirements_type='linear' | 'qflra' | 'propositional'`.
- For an alternative that *encourages* (rather than guarantees) satisfaction during
  training, see `shield_loss.ipynb`.